# AIFlow — R2 đã chọn → fine-tune giọng → My Drive → API

Chọn **Runtime → Change runtime type → GPU**. Mở ZIP do AIFlow xuất sau khi bạn chọn file.
Notebook không quét thêm file và không tự tải media khi khởi động.

**Không Run all.** Chạy từng bước, dừng ở bước duyệt dataset và nghe mẫu.
Lần sau chỉ cần các ô **1, 2, 8, 9, 10** để load giọng đã duyệt và bật API.
Tạo Colab Secrets `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` (Object Read đúng bucket)
và bật quyền Notebook access. Chỉ cần các secret này khi tải dữ liệu lần đầu.

Code VieNeu được ghim commit; model gốc và ASR được ghim revision trên Drive khi dùng lần đầu.
Không chia sẻ output token. Bật **Omit code cell output when saving**.

## 1. Upload gói lựa chọn (chưa tải media)

In [ ]:
from google.colab import files, drive
from pathlib import Path
import io, json, os, sys, zipfile, subprocess, secrets
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload đúng một ZIP từ trang Huấn luyện giọng của AIFlow.')
bundle_root = Path('/content/aiflow-voice-bundle')
bundle_root.mkdir(exist_ok=True)
required = {'selection.json', 'voice_training.py', 'audio_worker.py', 'finetune_r2.ipynb', 'train_resumable.py', 'training_control.py', 'control_voice_training.ipynb'}
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    if set(archive.namelist()) != required or any(f.file_size > 2_000_000 for f in archive.infolist()):
        raise ValueError('Không đúng gói AIFlow. Tạo lại gói từ trang Huấn luyện giọng.')
    for name in required:
        (bundle_root / name).write_bytes(archive.read(name))
del uploaded
drive.mount('/content/drive')
sys.path.insert(0, str(bundle_root))
import voice_training as training
job = training.open_selection(bundle_root / 'selection.json')
os.chdir('/content')
os.environ['HF_HOME'] = str(training.ROOT / 'hf-cache')
os.environ['AIFLOW_WORKER_STORAGE'] = '/content/drive/MyDrive/AIFlow/audio-worker'
os.environ['AIFLOW_WORKER_TOKEN'] = secrets.token_urlsafe(48)
print('Đã mount Drive. Chưa tải media và chưa load model.')

## 2. Cài môi trường trên Colab

In [ ]:
training.run('apt-get', '-qq', 'update')
training.run('apt-get', '-qq', '-y', 'install', 'ffmpeg', 'espeak-ng', 'git')
if not training.UPSTREAM.exists():
    training.run('git', 'clone', 'https://github.com/pnnbao97/VieNeu-TTS.git', training.UPSTREAM)
training.run('git', '-C', training.UPSTREAM, 'checkout', '--detach', training.UPSTREAM_COMMIT)
training.run(sys.executable, '-m', 'pip', 'install', str(training.UPSTREAM) + '[finetune]',
             'boto3>=1.35,<2', 'faster-whisper==1.1.1', 'av==18.0.0', 'ipywidgets>=8,<9',
             'fastapi>=0.115.12,<1', 'uvicorn>=0.34.2,<1', 'python-multipart>=0.0.12')
# LoRA uses ordinary FP32 weights; old optional torchao bundled by Colab breaks PEFT dispatch.
training.run(sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao')
training.gpu()
print('Môi trường đã cài. Nếu Colab yêu cầu restart, restart rồi chạy lại ô 1.')

## 3. Tải đúng file đã chọn

Chạy ô này mới bắt đầu đọc nội dung R2. File thay đổi từ lúc quét sẽ bị từ chối.
Chạy lại sẽ dùng file đã tải đúng checksum; không tự thêm file khác trong bucket.

In [ ]:
training.download_selected(job)

## 4. Tách audio và chép lời

Whisper chạy trên GPU Colab, FFmpeg tách audio; chỉ lấy đoạn 1–20 giây.
Không tự tách danh tính người nói. Nếu nguồn có nhiều người, bỏ các đoạn sai người ở bước tiếp theo.
Quá trình ASR có thể sai tên riêng/dấu câu: bắt buộc nghe và sửa trước khi train.

In [ ]:
training.prepare_clips(job)

## 5. Nghe, sửa, duyệt dataset

Chọn từng đoạn, nghe audio, sửa lời chép, bỏ chọn nếu không dùng.
Bấm **Lưu / đã nghe đoạn** trước khi chuyển đoạn. Sau cùng bấm **Duyệt dataset**.
Nên có 10–30 phút giọng sạch, nhiều kiểu câu. Ít nhất 20 đoạn được duyệt để có tập đánh giá.

In [ ]:
training.review_dataset(job)

## 6. Fine-tune LoRA thực sự và ghi model vào Drive

Chỉ chạy sau khi duyệt dataset. Mỗi lần chạy tạo một phiên bản mới, giữ adapter/checkpoint/merged model.
Nếu Colab ngắt giữa chừng, chạy lại tạo lượt mới; Bản trainer mới lưu đầy đủ optimizer/scheduler; có thể resume từ trang điều khiển hoặc training.train(job, epochs, resume_run=run_id).
Batch 1 + gradient checkpointing giúp giảm VRAM, nhưng GPU được cấp vẫn có thể không đủ.
Không chạy bước này trong lúc đang bật API.

In [ ]:
if 'worker' in globals() and worker.PIPELINE is not None:
    raise RuntimeError('Đang giữ model để phục vụ API. Dừng runtime và mở lại trước khi train.')
epochs = 3 # @param {type:"integer"}
run_dir = training.train(job, epochs=epochs)
print('Đã train và lưu:', run_dir)

## 7. Sinh mẫu, nghe và duyệt giọng

In [ ]:
sample_text = "Xin chào, đây là bản thử giọng sau huấn luyện. Hôm nay chúng ta bắt đầu một câu chuyện mới." # @param {type:"string"}
training.sample_voice(run_dir, sample_text)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
heard = widgets.Checkbox(description='Tôi đã nghe mẫu, đúng giọng và chất lượng dùng được', style={'description_width': 'initial'})
approve_button = widgets.Button(description='Duyệt giọng trên Drive')
approval_output = widgets.Output()
def approve_clicked(_):
    with approval_output:
        approval_output.clear_output()
        if not heard.value:
            print('Nghe mẫu rồi đánh dấu xác nhận trước.')
            return
        training.approve_voice(run_dir)
approve_button.on_click(approve_clicked)
display(heard, approve_button, approval_output)

## 8. Chọn giọng đã lưu trên My Drive

Chỉ hiển thị những phiên bản đã duyệt. Mỗi phiên API load một model/giọng để tiết kiệm VRAM.
Có thể có nhiều giọng trên Drive; đổi giọng bằng runtime mới và kết nối lại AIFlow.
Nếu ngắt sau train nhưng trước duyệt: dùng phần khôi phục cuối notebook để chọn bản nháp và chạy bước 7.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
choices = training.saved_voices()
if not choices:
    raise RuntimeError('Chưa có giọng đã duyệt. Hoàn tất bước 7 trước.')
voice_picker = widgets.Dropdown(options=choices, description='Giọng:', layout={'width': '95%'})
display(voice_picker)

## 9. Load giọng đã chọn và bật API / Cloudflare Tunnel

In [ ]:
import threading, urllib.request, uvicorn
if 'server' in globals() and not server.should_exit:
    raise RuntimeError('API đang chạy. Dùng ô dừng trước khi bật lại.')
if 'worker' not in globals() or worker.PIPELINE is None:
    worker = training.configure_worker(voice_picker.value)
else:
    selected_profile = training.read_json(Path(voice_picker.value) / 'profile.json')
    if selected_profile['id'] not in worker.VOICES:
        raise RuntimeError('Muốn đổi model: khởi động runtime mới và chọn lại giọng.')
worker.TOKEN = os.environ['AIFLOW_WORKER_TOKEN']
server = uvicorn.Server(uvicorn.Config(worker.app, host='127.0.0.1', port=8000,
                                      log_level='warning', access_log=False))
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o700)
tunnel_log = open('/content/aiflow-voice-tunnel.log', 'w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
print('Đã khởi động. Chạy ô lấy URL tiếp theo.')

## 10. Lấy URL và token → dán vào AIFlow → Kiểm tra → Lưu

In [ ]:
import re, time
url = None
for _ in range(30):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/aiflow-voice-tunnel.log').read_text())
    if match and server.started:
        url = match.group(0)
        break
    time.sleep(1)
if not url:
    raise RuntimeError('Chưa có URL hoặc API chưa sẵn sàng. Xem log tunnel, sau đó chạy lại ô này.')
print('URL:', url)
print('TOKEN:', worker.TOKEN)
print('Giọng:', worker.voices()[0]['name'])
print('Dán vào AIFlow /connections, kiểm tra và lưu. Không chia sẻ output token.')

## 11. Dừng API khi xong

In [ ]:
if 'worker' in globals() and worker.ACTIVE:
    raise RuntimeError('Còn tác vụ. Chờ hoàn tất hoặc hủy ở AIFlow trước khi dừng.')
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=15)
if 'server' in globals():
    server.should_exit = True
    api_thread.join(timeout=10)
if 'tunnel_log' in globals():
    tunnel_log.close()
print('Tắt API xong: Runtime → Disconnect and delete runtime để ngừng dùng GPU.')

## Khôi phục bản đã train nhưng chưa duyệt (tùy chọn)

Chạy ô này sau bước 1–2 của runtime mới, chọn phiên bản rồi chạy ô gán `run_dir`.
Quay lại bước 7 để nghe / duyệt. Chỉ những lượt đã train hoàn tất mới xuất hiện.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
drafts = training.saved_voices(approved_only=False)
if not drafts:
    raise RuntimeError('Chưa có lượt train hoàn tất.')
draft_picker = widgets.Dropdown(options=drafts, description='Bản lưu:', layout={'width': '95%'})
display(draft_picker)

In [ ]:
run_dir = Path(draft_picker.value)
print('Đã chọn', run_dir, '— quay lại bước 7.')

### Tài liệu và giới hạn

- [VieNeu fine-tune chính thức](https://github.com/pnnbao97/VieNeu-TTS/blob/c1390abbdb2eedcdf58eafb546966c06ce27af71/finetune/README.md)
- [Cloudflare R2 token](https://developers.cloudflare.com/r2/api/tokens/)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html): tài nguyên/GPU phụ thuộc phiên, không dùng keep-alive.
- My Drive: `AIFlow/voice-training/<job>/runs/<run>/` có adapter, model merged, profile, sample và log training.
- Không cung cấp HTTP train job/status; các bước tải/duyệt/train diễn ra trong notebook.
- Không tự nhận diện nhiều người nói, không huấn luyện giọng từ nhóm file chưa được bạn chọn.